In [11]:
from scapy.all import rdpcap, PcapReader

# 方式1: 一次性读取所有数据包(适合小文件)
packets = rdpcap('../data/apollo_eth0_sample_94.pcap')
print(f"总共有 {len(packets)} 个数据包")

总共有 566 个数据包


In [24]:
packets[93][DNS].an

[<DNSRR  rrname=b'www.hestia.com.au.' type=A cacheflush=0 rclass=IN ttl=604800 rdata=54.185.11.51 |>]

In [ ]:
from scapy.all import rdpcap, IP, TCP, UDP, ICMP, DNS, DHCP, Raw
from scapy.layers.http import HTTPRequest, HTTPResponse
import ssl

def load_pcap(file_path):
    packets = rdpcap(file_path)
    return packets

def layer_detector(packet):
    """detect layers in a packet, and add all the info into a list"""
    target_layers=[IP, TCP, UDP, ICMP, DNS, DHCP, HTTPRequest, HTTPResponse, Raw]
    packet_info=[]
    for layer in target_layers:
        if packet.haslayer(layer):
            icmp_layer = packet[layer]
            packet_info.append(icmp_layer.show())
            #deal stmp
            if layer==TCP and (packet[TCP].dport==25 or packet[TCP].sport==25):
                stmp_info = [
                    "*** STMP Layer ***",
                    f"Source Port: {packet[TCP].sport}",
                    f"Destination Port: {packet[TCP].dport}",
                    f"Raw Data: {packet[Raw].load.decode('utf-8', 'ignore')}"
                ]
                packet_info.append(stmp_info)
    
    # 添加raw层的内容
    if packet.haslayer(Raw):
        raw_data = packet[Raw].load
        try:
            decoded_data = raw_data.decode('utf-8')
        except UnicodeDecodeError:
            decoded_data = raw_data.decode('utf-8', 'ignore')
        packet_info.append(f"*** Raw Layer ***\nData: {decoded_data}\n")

    return packet_info

layer_detector(packets[253])

def layer_simple_detector(packet):
    """detect layers in a packet, and add all the info into a list"""
    #target_layers=[IP, TCP, UDP, ICMP, DNS, DHCP, HTTPRequest, HTTPResponse, Raw]
    packet_info=[]
    packet_info.append(f"*** Packet Summary ***\n{packet.summary()}\n")
    if packet.haslayer(IP):
        packet_info.append(f"*** IP Layer ***\nSource IP: {packet[IP].src}\nDestination IP: {packet[IP].dst}\n")
    if packet.haslayer(TCP): # 输出tcp的端口, 是ACK还是SYN还是啥, 长度
        packet_info.append(f"*** TCP Layer ***\nSource Port: {packet[TCP].sport}\nDestination Port: {packet[TCP].dport}\nFlags: {packet[TCP].flags}\nLength: {len(packet[TCP])}\n")
    if packet.haslayer(UDP): # 输出udp的端口
        packet_info.append(f"*** UDP Layer ***\nSource Port: {packet[UDP].sport}\nDestination Port: {packet[UDP].dport}\n")
    if packet.haslayer(ICMP): # 输出icmp的type和code
        packet_info.append(f"*** ICMP Layer ***\nType: {packet[ICMP].type}\nCode: {packet[ICMP].code}\n")
    if packet.haslayer(DNS): # 输出dns的id和查询名, 如果是相应也输出响应
        packet_info.append(f"*** DNS Layer ***\nID: {packet[DNS].id}\nQuery Name: {packet[DNS].qd.qname.decode('utf-8')}\n")
        if packet[DNS].ancount > 0:
            packet_info.append(f"Answers: {packet[DNS].an}\n")
    if packet.haslayer(DHCP): # 输出dhcp的options
        packet_info.append(f"*** DHCP Layer ***\nOptions: {packet[DHCP].options}\n")
    if packet.haslayer(HTTPRequest): # 输出http请求的host, path, method, 包大小
        packet_info.append(f"*** HTTP Request Layer ***\nHost: {packet[HTTPRequest].Host.decode('utf-8')}\nPath: {packet[HTTPRequest].Path.decode('utf-8')}\nMethod: {packet[HTTPRequest].Method.decode('utf-8')}\nPacket Size: {len(packet[HTTPRequest])}\n")
    if packet.haslayer(HTTPResponse): # 输出http响应的状态码
        packet_info.append(f"*** HTTP Response Layer ***\nStatus Code: {packet[HTTPResponse].Status_Code.decode('utf-8')}\nPacket Size: {len(packet[HTTPResponse])}\n")
    if packet.haslayer(TCP) and (packet[TCP].dport==25 or packet[TCP].sport==25): # STMP
        packet_info.append(f"*** STMP Layer ***\nSource Port: {packet[TCP].sport}\nDestination Port: {packet[TCP].dport}\nRaw Data: {packet[Raw].load.decode('utf-8', 'ignore')}\n")

    return packet_info

def split_pcap_to_txt(packets, session_id, txt_split_num=5, simple_mode=False):
    """load every packets, using layer_detector, and split them into several txt files"""
    total_packets = len(packets)
    packets_per_file = total_packets // txt_split_num + 1
    txt_file_paths = []
    for i in range(txt_split_num):
        start_index = i * packets_per_file
        end_index = min((i + 1) * packets_per_file, total_packets)
        if start_index >= total_packets:
            break
        txt_file_path = f"{session_id}_part{i+1}.txt"
        with open(txt_file_path, 'w') as txt_file:
            for j in range(start_index, end_index):
                if simple_mode:
                    packet_info = layer_simple_detector(packets[j])
                else:
                    packet_info = layer_detector(packets[j])
                txt_file.write(f"*** Packet {j+1} ***\n")
                for info in packet_info:
                    if isinstance(info, list):
                        for line in info:
                            txt_file.write(f"{line}\n")
                    else:
                        txt_file.write(f"{info}\n")
                txt_file.write("\n\n")
        txt_file_paths.append(txt_file_path)
    return txt_file_paths




###[ IP ]###
  version   = 4
  ihl       = 5
  tos       = 0x0
  len       = 211
  id        = 34121
  flags     = DF
  frag      = 0
  ttl       = 64
  proto     = tcp
  chksum    = 0x58e1
  src       = 133.184.40.119
  dst       = 133.184.40.19
  \options   \
###[ TCP ]###
     sport     = 43212
     dport     = http
     seq       = 3623821412
     ack       = 2859063475
     dataofs   = 8
     reserved  = 0
     flags     = PA
     window    = 501
     chksum    = 0x5cc0
     urgptr    = 0
     options   = [('NOP', None), ('NOP', None), ('Timestamp', (3577219618, 3491263258))]
###[ HTTP 1 ]###
###[ HTTP Request ]###
        Method    = GET
        Path      = /robots.txt
        Http_Version= HTTP/1.1
        A_IM      = None
        Accept    = */*
        Accept_Charset= None
        Accept_Datetime= None
        Accept_Encoding= identity
        Accept_Language= None
        Access_Control_Request_Headers= None
        Access_Control_Request_Method= None
        Authorization= N

[]

In [9]:
from scapy.all import rdpcap, IP, TCP, UDP, ICMP, DNS, DHCP, Raw
from scapy.layers.http import HTTPRequest, HTTPResponse
import ssl

# 读取pcap文件
packets = rdpcap('../data/apollo_eth0_sample_94.pcap')

# 1. ICMP协议解析
def parse_icmp(packets):
    print("\n=== ICMP 数据包 ===")
    icmp_count = 0
    for pkt in packets:
        if pkt.haslayer(ICMP):
            icmp_count += 1
            icmp_layer = pkt[ICMP]
            print(f"ICMP类型: {icmp_layer.type}, 代码: {icmp_layer.code}")
            
            # ICMP类型说明
            if icmp_layer.type == 8:
                print("  -> Echo Request (Ping请求)")
            elif icmp_layer.type == 0:
                print("  -> Echo Reply (Ping响应)")
            elif icmp_layer.type == 3:
                print("  -> Destination Unreachable")
            elif icmp_layer.type == 11:
                print("  -> Time Exceeded")
                
            if pkt.haslayer(IP):
                print(f"  源IP: {pkt[IP].src} -> 目标IP: {pkt[IP].dst}")
    
    print(f"总共找到 {icmp_count} 个ICMP数据包")

# 2. TCP协议解析
def parse_tcp(packets):
    print("\n=== TCP 数据包 ===")
    tcp_count = 0
    tcp_connections = {}
    
    for pkt in packets:
        if pkt.haslayer(TCP):
            tcp_count += 1
            tcp_layer = pkt[TCP]
            
            # 获取连接信息
            if pkt.haslayer(IP):
                src_ip = pkt[IP].src
                dst_ip = pkt[IP].dst
                src_port = tcp_layer.sport
                dst_port = tcp_layer.dport
                
                # 记录连接
                conn_key = f"{src_ip}:{src_port} -> {dst_ip}:{dst_port}"
                if conn_key not in tcp_connections:
                    tcp_connections[conn_key] = {
                        'flags': [],
                        'seq': [],
                        'ack': []
                    }
                
                # 解析TCP标志位
                flags = []
                if tcp_layer.flags & 0x02: flags.append('SYN')
                if tcp_layer.flags & 0x10: flags.append('ACK')
                if tcp_layer.flags & 0x01: flags.append('FIN')
                if tcp_layer.flags & 0x04: flags.append('RST')
                if tcp_layer.flags & 0x08: flags.append('PSH')
                
                tcp_connections[conn_key]['flags'].extend(flags)
                tcp_connections[conn_key]['seq'].append(tcp_layer.seq)
                tcp_connections[conn_key]['ack'].append(tcp_layer.ack)
    
    print(f"总共找到 {tcp_count} 个TCP数据包")
    print(f"发现 {len(tcp_connections)} 个TCP连接:")
    for conn, info in list(tcp_connections.items())[:5]:  # 只显示前5个
        print(f"  {conn}")
        unique_flags = set(info['flags'])
        print(f"    标志位: {', '.join(unique_flags)}")

# 3. DNS协议解析
def parse_dns(packets):
    print("\n=== DNS 数据包 ===")
    dns_count = 0
    dns_queries = []
    dns_responses = []
    
    for pkt in packets:
        if pkt.haslayer(DNS):
            dns_count += 1
            dns_layer = pkt[DNS]
            
            # DNS查询
            if dns_layer.qr == 0:  # 0表示查询
                if dns_layer.qd:  # 查询部分
                    for query in dns_layer.qd:
                        query_name = query.qname.decode() if isinstance(query.qname, bytes) else query.qname
                        dns_queries.append(query_name)
                        print(f"DNS查询: {query_name}")
            
            # DNS响应
            elif dns_layer.qr == 1:  # 1表示响应
                if dns_layer.an:  # 回答部分
                    for answer in dns_layer.an:
                        if hasattr(answer, 'rdata'):
                            dns_responses.append(str(answer.rdata))
                            print(f"DNS响应: {answer.rdata}")
    
    print(f"总共找到 {dns_count} 个DNS数据包")
    print(f"  查询: {len(dns_queries)} 个")
    print(f"  响应: {len(dns_responses)} 个")

# 4. DHCP协议解析
def parse_dhcp(packets):
    print("\n=== DHCP 数据包 ===")
    dhcp_count = 0
    dhcp_types = {
        1: 'DISCOVER',
        2: 'OFFER',
        3: 'REQUEST',
        4: 'DECLINE',
        5: 'ACK',
        6: 'NAK',
        7: 'RELEASE',
        8: 'INFORM'
    }
    
    for pkt in packets:
        if pkt.haslayer(DHCP):
            dhcp_count += 1
            dhcp_layer = pkt[DHCP]
            
            # 获取DHCP消息类型
            for option in dhcp_layer.options:
                if option[0] == 'message-type':
                    msg_type = option[1]
                    msg_type_name = dhcp_types.get(msg_type, 'UNKNOWN')
                    print(f"DHCP消息类型: {msg_type_name} ({msg_type})")
                elif option[0] == 'requested_addr':
                    print(f"  请求的IP地址: {option[1]}")
                elif option[0] == 'server_id':
                    print(f"  DHCP服务器: {option[1]}")
                elif option[0] == 'hostname':
                    print(f"  主机名: {option[1]}")
    
    print(f"总共找到 {dhcp_count} 个DHCP数据包")

# 5. HTTPS协议解析 (通过端口443识别)
def parse_https(packets):
    print("\n=== HTTPS/TLS 数据包 ===")
    https_count = 0
    tls_handshakes = 0
    
    for pkt in packets:
        if pkt.haslayer(TCP):
            tcp_layer = pkt[TCP]
            
            # 检查是否是443端口（HTTPS默认端口）
            if tcp_layer.sport == 443 or tcp_layer.dport == 443:
                https_count += 1
                
                # 检查是否有负载数据
                if pkt.haslayer(Raw):
                    payload = bytes(pkt[Raw])
                    
                    # 检查TLS记录
                    if len(payload) > 5:
                        # TLS记录格式: Content Type (1 byte) + Version (2 bytes) + Length (2 bytes)
                        content_type = payload[0]
                        
                        # TLS内容类型
                        if content_type == 22:  # Handshake
                            tls_handshakes += 1
                            print(f"TLS握手消息")
                        elif content_type == 23:  # Application Data
                            print(f"TLS应用数据（加密）")
                        elif content_type == 21:  # Alert
                            print(f"TLS警告消息")
                        elif content_type == 20:  # ChangeCipherSpec
                            print(f"TLS密码规格变更")
    
    print(f"总共找到 {https_count} 个HTTPS相关数据包")
    print(f"  TLS握手: {tls_handshakes} 个")

# 6. SMTP协议解析 (通过端口25/587/465识别)
def parse_smtp(packets):
    print("\n=== SMTP 数据包 ===")
    smtp_count = 0
    smtp_commands = []
    smtp_ports = [25, 587, 465]  # SMTP常用端口
    
    for pkt in packets:
        if pkt.haslayer(TCP):
            tcp_layer = pkt[TCP]
            
            # 检查SMTP端口
            if tcp_layer.sport in smtp_ports or tcp_layer.dport in smtp_ports:
                if pkt.haslayer(Raw):
                    smtp_count += 1
                    payload = pkt[Raw].load
                    
                    try:
                        # 尝试解码为文本
                        text = payload.decode('utf-8', errors='ignore')
                        
                        # 检查SMTP命令
                        smtp_keywords = ['HELO', 'EHLO', 'MAIL FROM', 'RCPT TO', 
                                        'DATA', 'QUIT', '220', '250', '354', '221']
                        
                        for keyword in smtp_keywords:
                            if keyword in text:
                                smtp_commands.append(text[:100])  # 只保存前100个字符
                                print(f"SMTP: {text[:100].strip()}")
                                break
                    except:
                        pass
    
    print(f"总共找到 {smtp_count} 个SMTP相关数据包")
    print(f"  识别出 {len(smtp_commands)} 个SMTP命令/响应")

# 执行所有解析函数
def analyze_all_protocols(packets):
    print(f"\n开始分析 {len(packets)} 个数据包...\n")
    print("=" * 50)
    
    parse_icmp(packets)
    parse_tcp(packets)
    parse_dns(packets)
    parse_dhcp(packets)
    parse_https(packets)
    parse_smtp(packets)
    
    print("\n" + "=" * 50)
    print("协议分析完成！")

# 运行分析
analyze_all_protocols(packets)

# 额外：统计协议分布
def protocol_statistics(packets):
    print("\n=== 协议统计 ===")
    protocols = {
        'TCP': 0,
        'UDP': 0,
        'ICMP': 0,
        'DNS': 0,
        'DHCP': 0,
        'HTTP': 0,
        'HTTPS': 0,
        'SMTP': 0
    }
    
    for pkt in packets:
        if pkt.haslayer(TCP):
            protocols['TCP'] += 1
            tcp = pkt[TCP]
            if tcp.dport == 443 or tcp.sport == 443:
                protocols['HTTPS'] += 1
            elif tcp.dport == 80 or tcp.sport == 80:
                protocols['HTTP'] += 1
            elif tcp.dport in [25, 587, 465] or tcp.sport in [25, 587, 465]:
                protocols['SMTP'] += 1
        
        if pkt.haslayer(UDP):
            protocols['UDP'] += 1
        
        if pkt.haslayer(ICMP):
            protocols['ICMP'] += 1
        
        if pkt.haslayer(DNS):
            protocols['DNS'] += 1
        
        if pkt.haslayer(DHCP):
            protocols['DHCP'] += 1
    
    print("\n协议分布:")
    for proto, count in protocols.items():
        if count > 0:
            percentage = (count / len(packets)) * 100
            print(f"  {proto}: {count} 包 ({percentage:.2f}%)")

protocol_statistics(packets)


开始分析 566 个数据包...


=== ICMP 数据包 ===
ICMP类型: 8, 代码: 0
  -> Echo Request (Ping请求)
  源IP: 133.184.40.222 -> 目标IP: 133.184.40.119
ICMP类型: 0, 代码: 0
  -> Echo Reply (Ping响应)
  源IP: 133.184.40.119 -> 目标IP: 133.184.40.222
ICMP类型: 8, 代码: 0
  -> Echo Request (Ping请求)
  源IP: 133.184.40.119 -> 目标IP: 133.184.40.118
ICMP类型: 0, 代码: 0
  -> Echo Reply (Ping响应)
  源IP: 133.184.40.118 -> 目标IP: 133.184.40.119
ICMP类型: 8, 代码: 0
  -> Echo Request (Ping请求)
  源IP: 133.184.40.119 -> 目标IP: 133.184.40.118
ICMP类型: 0, 代码: 0
  -> Echo Reply (Ping响应)
  源IP: 133.184.40.118 -> 目标IP: 133.184.40.119
ICMP类型: 3, 代码: 3
  -> Destination Unreachable
  源IP: 133.184.40.119 -> 目标IP: 54.185.11.39
ICMP类型: 8, 代码: 0
  -> Echo Request (Ping请求)
  源IP: 133.184.40.119 -> 目标IP: 133.184.40.118
ICMP类型: 0, 代码: 0
  -> Echo Reply (Ping响应)
  源IP: 133.184.40.118 -> 目标IP: 133.184.40.119
ICMP类型: 8, 代码: 0
  -> Echo Request (Ping请求)
  源IP: 133.184.40.119 -> 目标IP: 133.184.40.118
ICMP类型: 0, 代码: 0
  -> Echo Reply (Ping响应)
  源IP: 133.184.40.118 -> 目标IP

In [8]:
print(packets[247].show())

###[ Ethernet ]###
  dst       = 24:a0:74:26:fb:a5
  src       = 00:13:21:83:2a:5a
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 60
     id        = 0
     flags     = DF
     frag      = 0
     ttl       = 64
     proto     = tcp
     chksum    = 0xdec1
     src       = 133.184.40.19
     dst       = 133.184.40.119
     \options   \
###[ TCP ]###
        sport     = http
        dport     = 43212
        seq       = 2859062979
        ack       = 3623821263
        dataofs   = 10
        reserved  = 0
        flags     = SA
        window    = 65160
        chksum    = 0x5c29
        urgptr    = 0
        options   = [('MSS', 1460), ('SAckOK', b''), ('Timestamp', (3491263257, 3577219614)), ('NOP', None), ('WScale', 7)]

None


In [5]:
from scapy.all import rdpcap, Packet
import json

def packet_to_dict(pkt):
    """递归提取数据包的所有层和字段"""
    packet_dict = {
        'summary': pkt.summary(),
        'timestamp': float(pkt.time) if hasattr(pkt, 'time') else None,
        'length': len(pkt),
        'layers': []
    }
    
    # 遍历所有协议层
    layer = pkt
    while layer:
        layer_dict = {
            'layer_name': layer.name,
            'fields': {}
        }
        
        # 提取该层的所有字段
        for field_name, field_value in layer.fields.items():
            # 处理不同类型的字段值
            if isinstance(field_value, bytes):
                layer_dict['fields'][field_name] = field_value.hex()
            elif isinstance(field_value, Packet):
                # 如果字段值是另一个数据包，递归处理
                layer_dict['fields'][field_name] = packet_to_dict(field_value)
            else:
                layer_dict['fields'][field_name] = str(field_value)
        
        packet_dict['layers'].append(layer_dict)
        layer = layer.payload if layer.payload else None
    
    return packet_dict

# 读取并转换
packets = rdpcap('apollo_eth0_sample_0.pcap')
packets_json = []

for i, pkt in enumerate(packets):
    packet_data = packet_to_dict(pkt)
    packet_data['packet_index'] = i
    packets_json.append(packet_data)

# 保存为 JSON 文件
with open('packets.json', 'w', encoding='utf-8') as f:
    json.dump(packets_json, f, indent=2, ensure_ascii=False)

print(f"成功转换 {len(packets_json)} 个数据包")

成功转换 563 个数据包


In [ ]:
from scapy.all import rdpcap, Packet
import json
import math

def packet_to_dict(pkt):
    """递归提取数据包的所有层和字段"""
    packet_dict = {
        'summary': pkt.summary(),
        'timestamp': float(pkt.time) if hasattr(pkt, 'time') else None,
        'length': len(pkt),
        'layers': []
    }
    
    # 遍历所有协议层
    layer = pkt
    while layer:
        layer_dict = {
            'layer_name': layer.name,
            'fields': {}
        }
        
        # 提取该层的所有字段
        for field_name, field_value in layer.fields.items():
            # 处理不同类型的字段值
            if isinstance(field_value, bytes):
                layer_dict['fields'][field_name] = field_value.hex()
            elif isinstance(field_value, Packet):
                # 如果字段值是另一个数据包，递归处理
                layer_dict['fields'][field_name] = packet_to_dict(field_value)
            else:
                layer_dict['fields'][field_name] = str(field_value)
        
        packet_dict['layers'].append(layer_dict)
        layer = layer.payload if layer.payload else None
    
    return packet_dict

# 读取数据包
packets = rdpcap('apollo_eth0_sample_0.pcap')
total_packets = len(packets)
num_files = 10

# 计算每个文件应包含的数据包数量
packets_per_file = math.ceil(total_packets / num_files)

print(f"总共 {total_packets} 个数据包，将拆分为 {num_files} 个文件")
print(f"每个文件约 {packets_per_file} 个数据包")

filename_list=[]
# 拆分并保存
for file_index in range(num_files):
    start_idx = file_index * packets_per_file
    end_idx = min(start_idx + packets_per_file, total_packets)
    
    # 如果起始索引已经超出范围，跳出循环
    if start_idx >= total_packets:
        break
    
    packets_json = []
    for i in range(start_idx, end_idx):
        packet_data = packet_to_dict(packets[i])
        packet_data['packet_index'] = i  # 保持原始索引
        packets_json.append(packet_data)
    
    # 保存为单独的 JSON 文件
    filename = f'packets_part_{file_index + 1:02d}.json'
    filename_list.append(filename)
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(packets_json, f, indent=2, ensure_ascii=False)
    
    print(f"✓ {filename}: 包含数据包 {start_idx}-{end_idx-1} ({len(packets_json)} 个)")

print(f"\n成功拆分完成！")

总共 563 个数据包，将拆分为 10 个文件
每个文件约 57 个数据包
✓ packets_part_01.json: 包含数据包 0-56 (57 个)
✓ packets_part_02.json: 包含数据包 57-113 (57 个)
✓ packets_part_03.json: 包含数据包 114-170 (57 个)
✓ packets_part_04.json: 包含数据包 171-227 (57 个)
✓ packets_part_05.json: 包含数据包 228-284 (57 个)
✓ packets_part_06.json: 包含数据包 285-341 (57 个)
✓ packets_part_07.json: 包含数据包 342-398 (57 个)
✓ packets_part_08.json: 包含数据包 399-455 (57 个)
✓ packets_part_09.json: 包含数据包 456-512 (57 个)
✓ packets_part_10.json: 包含数据包 513-562 (50 个)

成功拆分完成！


In [ ]:
# 初始化 OpenAI 客户端
client = OpenAI()  # 确保设置了 OPENAI_API_KEY 环境变量

uploaded_file_ids = []

for filename in output_files:
    try:
        print(f"正在上传 {filename}...")
        with open(filename, 'rb') as f:
            file_response = client.files.create(
                file=f,
                purpose='assistants'  # 用于 assistants/vector store
            )
        
        uploaded_file_ids.append(file_response.id)
        print(f"✓ {filename} 上传成功! File ID: {file_response.id}")
        
    except Exception as e:
        print(f"✗ {filename} 上传失败: {str(e)}")

print(f"\n成功上传 {len(uploaded_file_ids)} 个文件！\n")

# ========== 步骤3: 创建或使用现有 Vector Store ==========
print("=" * 60)
print("步骤3: 创建 Vector Store")
print("=" * 60)

# 选项A: 创建新的 Vector Store
try:
    vector_store = client.vector_stores.create(
        name="PCAP Packets Analysis",
        file_ids=[]  # 先创建空的，稍后用 batch 添加文件
    )
    vector_store_id = vector_store.id
    print(f"✓ Vector Store 创建成功! ID: {vector_store_id}\n")
    
except Exception as e:
    print(f"✗ Vector Store 创建失败: {str(e)}\n")
    exit(1)

# 选项B: 如果你已有 Vector Store，可以直接使用
# vector_store_id = "vs_abc123"  # 替换为你的 Vector Store ID
# print(f"使用现有 Vector Store: {vector_store_id}\n")

# ========== 步骤4: 创建 File Batch ==========
print("=" * 60)
print("步骤4: 创建 File Batch")
print("=" * 60)

try:
    # 创建 file batch
    file_batch = client.vector_stores.file_batches.create(
        vector_store_id=vector_store_id,
        file_ids=uploaded_file_ids
    )
    
    print(f"✓ File Batch 创建成功! ID: {file_batch.id}")
    print(f"状态: {file_batch.status}")
    print(f"文件统计: {file_batch.file_counts}\n")
    
    # 等待 batch 处理完成
    print("等待文件处理完成...")
    while file_batch.status in ['in_progress', 'cancelling']:
        time.sleep(5)
        file_batch = client.vector_stores.file_batches.retrieve(
            vector_store_id=vector_store_id,
            batch_id=file_batch.id
        )
        print(f"当前状态: {file_batch.status}, 文件统计: {file_batch.file_counts}")
    
    print(f"\n✓ Batch 处理完成! 最终状态: {file_batch.status}")
    
except Exception as e:
    print(f"✗ File Batch 创建失败: {str(e)}")
    exit(1)